# 03 — Exploratory Data Analysis

**AttriSense · Employee Attrition Prediction & Analytics System**

---

## Purpose

Exploratory analysis translates raw workforce data into HR decisions. Each chart in this notebook answers a specific business question — nothing is plotted for decoration.

**Data source:** `data/processed/employee_attrition_cleaned.parquet` (output of `02_Data_Preprocessing.ipynb`)

**Guiding questions:**
- Where is attrition concentrated (department, role, overtime)?
- Which employee profiles leave more often (age, income)?
- How do satisfaction scores relate to departure?
- Which numeric factors correlate with attrition risk?

## Setup

Load the cleaned dataset and define shared plotting constants. All rates are computed from the data — not assumed.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import seaborn as sns

from attrisense.config import load_config
from attrisense.data import load_processed_data
from attrisense.utils.paths import REPORTS_FIGURES_DIR

config = load_config()
FIGURES_DIR = REPORTS_FIGURES_DIR
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Consistent palette
COLOR_STAY = "#2a9d8f"
COLOR_LEAVE = "#e76f51"
COLOR_RATE = "#264653"
OVERALL_RATE_COLOR = "#6c757d"

plt.style.use("seaborn-v0_8-whitegrid")
pio.templates.default = "plotly_white"

df = load_processed_data(config)
df["Attrition_Yes"] = (df[config.target_column] == config.positive_class).astype(int)

overall_rate = df["Attrition_Yes"].mean() * 100
n_total = len(df)
n_left = df["Attrition_Yes"].sum()

print(f"Employees : {n_total:,}")
print(f"Attrition : {n_left} ({overall_rate:.2f}%)")
print(f"Retained  : {n_total - n_left} ({100 - overall_rate:.2f}%)")

In [ ]:
def attrition_summary(data: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """Compute employee count and attrition rate (%) by group."""
    summary = (
        data.groupby(group_col, observed=True)
        .agg(
            employees=(config.target_column, "count"),
            attrition_rate=("Attrition_Yes", lambda s: s.mean() * 100),
        )
        .reset_index()
    )
    summary["attrition_rate"] = summary["attrition_rate"].round(2)
    return summary.sort_values("attrition_rate", ascending=False)

---

## 1. Distribution Plots

### Business question: *What do the age and income profiles of staying vs. leaving employees look like?*

Understanding whether leavers skew younger or lower-paid helps HR target retention budgets and compensation reviews.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, col, xlabel in zip(
    axes,
    ["Age", "MonthlyIncome"],
    ["Age (years)", "Monthly Income"],
):
    for label, stay in [(config.negative_class, True), (config.positive_class, False)]:
        subset = df[df[config.target_column] == label][col]
        color = COLOR_STAY if stay else COLOR_LEAVE
        ax.hist(
            subset,
            bins=20,
            alpha=0.55,
            density=True,
            label=f"{'Stayed' if stay else 'Left'} (n={len(subset)})",
            color=color,
            edgecolor="white",
            linewidth=0.5,
        )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    ax.set_title(f"Distribution of {col} by Attrition")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_age_income_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

age_left = df.loc[df["Attrition_Yes"] == 1, "Age"].median()
age_stay = df.loc[df["Attrition_Yes"] == 0, "Age"].median()
inc_left = df.loc[df["Attrition_Yes"] == 1, "MonthlyIncome"].median()
inc_stay = df.loc[df["Attrition_Yes"] == 0, "MonthlyIncome"].median()

print(f"Median age    — Left: {age_left:.0f} yrs | Stayed: {age_stay:.0f} yrs")
print(f"Median income — Left: ${inc_left:,.0f} | Stayed: ${inc_stay:,.0f}")

**Interpretation:** Employees who left are concentrated at younger ages and lower income levels. Median age among leavers is **4 years lower** than among those who stayed; median monthly income is roughly **$2,000 lower**. Compensation and early-career retention programs address a real pattern in the data, not a modeling artifact.

---

## 2. Count Plot — Overall Attrition Split

### Business question: *How imbalanced is the attrition problem?*

Class imbalance affects how we evaluate models and how HR should interpret risk scores.

In [ ]:
counts = df[config.target_column].value_counts().reindex([config.negative_class, config.positive_class])

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    ["Stayed", "Left"],
    [counts[config.negative_class], counts[config.positive_class]],
    color=[COLOR_STAY, COLOR_LEAVE],
    edgecolor="white",
)
ax.set_ylabel("Employee Count")
ax.set_title("Workforce Attrition Count")
ax.spines[["top", "right"]].set_visible(False)

for bar, val in zip(bars, counts.values):
    pct = val / n_total * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 15,
        f"{val:,}\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=10,
    )

plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_attrition_count.png", dpi=150, bbox_inches="tight")
plt.show()

**Interpretation:** Roughly **1 in 6 employees** left during the observation period. Predictive models must prioritize recall on the minority class — missing a leaver is costlier than a false alarm when planning retention outreach.

---

## 3. Boxplots — Age and Income by Attrition

### Business question: *How much do age and pay differ between employees who stay and those who leave?*

Boxplots show central tendency and spread — useful for spotting whether leavers are a distinct segment.

In [ ]:
plot_df = df.copy()
plot_df["Status"] = plot_df[config.target_column].map(
    {config.negative_class: "Stayed", config.positive_class: "Left"}
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

sns.boxplot(
    data=plot_df, x="Status", y="Age", order=["Stayed", "Left"],
    palette={"Stayed": COLOR_STAY, "Left": COLOR_LEAVE},
    ax=axes[0], width=0.5, fliersize=3,
)
axes[0].set_title("Age by Attrition Status")
axes[0].set_xlabel("")

sns.boxplot(
    data=plot_df, x="Status", y="MonthlyIncome", order=["Stayed", "Left"],
    palette={"Stayed": COLOR_STAY, "Left": COLOR_LEAVE},
    ax=axes[1], width=0.5, fliersize=3,
)
axes[1].set_title("Monthly Income by Attrition Status")
axes[1].set_xlabel("")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_age_income_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

**Interpretation:** The median income box for leavers sits noticeably lower, with a tighter upper quartile — senior high earners rarely appear in the attrition group. Age shows the same pattern: leavers cluster in the lower half of the age range. Pay equity reviews and career-pathing for junior staff are data-supported interventions.

---

## 4. Correlation Heatmap

### Business question: *Which numeric workforce attributes move together, and which align most with attrition?*

Pearson correlation on numeric features (excluding employee ID). Focus on the attrition row/column for actionable signals.

In [ ]:
numeric_cols = [
    c for c in df.select_dtypes(include="number").columns
    if c not in [config.id_column, "Attrition_Yes"]
]

corr = df[numeric_cols + ["Attrition_Yes"]].corr(numeric_only=True)

attrition_corr = corr["Attrition_Yes"]
if isinstance(attrition_corr, pd.DataFrame):
    attrition_corr = attrition_corr.iloc[:, 0]
attrition_corr = attrition_corr.drop("Attrition_Yes")

top_features = attrition_corr.abs().sort_values(ascending=False).head(12).index.tolist()
heatmap_cols = top_features + ["Attrition_Yes"]
corr_subset = corr.loc[heatmap_cols, heatmap_cols]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_subset,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-0.3,
    vmax=0.3,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8, "label": "Pearson r"},
    ax=ax,
)
ax.set_title("Correlation Among Top Attrition-Related Features")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

print("Strongest correlations with attrition:")
print(attrition_corr.reindex(top_features).round(3).to_string())

**Interpretation:** Attrition correlates most with **tenure and career-progression variables** (`TotalWorkingYears`, `JobLevel`, `YearsInCurrentRole`, `YearsWithCurrManager`) and **compensation** (`MonthlyIncome`, `StockOptionLevel`). These are negative correlations — employees with more tenure, higher job level, and higher pay leave less often. `DistanceFromHome` is the strongest *positive* correlate: longer commutes associate with higher departure rates.

---

## 5. Attrition by Department

### Business question: *Which departments bear the highest turnover burden?*

Department-level rates guide where HR partners should focus retention campaigns.

In [ ]:
dept = attrition_summary(df, "Department")

fig = px.bar(
    dept,
    x="Department",
    y="attrition_rate",
    text="attrition_rate",
    hover_data={"employees": True, "attrition_rate": ":.1f"},
    labels={"attrition_rate": "Attrition Rate (%)", "Department": ""},
    title="Attrition Rate by Department",
    color="attrition_rate",
    color_continuous_scale=["#2a9d8f", "#e76f51"],
    range_color=[dept["attrition_rate"].min(), dept["attrition_rate"].max()],
)
fig.add_hline(
    y=overall_rate, line_dash="dash", line_color=OVERALL_RATE_COLOR,
    annotation_text=f"Company avg: {overall_rate:.1f}%",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(showlegend=False, coloraxis_showscale=False, yaxis_range=[0, 28])
fig.write_html(FIGURES_DIR / "eda_attrition_by_department.html")
fig.show()

dept

**Interpretation:** **Sales** has the highest attrition rate (~21%), exceeding the company average by about 5 percentage points. **Research & Development** is lowest (~14%). Sales teams often face quota pressure and external job markets — targeted stay interviews in Sales should be a priority.

---

## 6. Attrition by Job Role

### Business question: *Which roles are hardest to retain?*

Role-level detail is more actionable than department alone — two roles in the same department can have very different turnover.

In [ ]:
role = attrition_summary(df, "JobRole").sort_values("attrition_rate", ascending=True)

fig = px.bar(
    role,
    x="attrition_rate",
    y="JobRole",
    orientation="h",
    text="attrition_rate",
    hover_data={"employees": True, "attrition_rate": ":.1f"},
    labels={"attrition_rate": "Attrition Rate (%)", "JobRole": ""},
    title="Attrition Rate by Job Role",
    color="attrition_rate",
    color_continuous_scale=["#2a9d8f", "#e76f51"],
    range_color=[role["attrition_rate"].min(), role["attrition_rate"].max()],
)
fig.add_vline(
    x=overall_rate, line_dash="dash", line_color=OVERALL_RATE_COLOR,
    annotation_text=f"Avg: {overall_rate:.1f}%",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(showlegend=False, coloraxis_showscale=False, height=420, xaxis_range=[0, 48])
fig.write_html(FIGURES_DIR / "eda_attrition_by_jobrole.html")
fig.show()

role.sort_values("attrition_rate", ascending=False)

**Interpretation:** **Sales Representative** attrition (~40%) is nearly **2.5× the company average** — the single highest-risk role. At the other end, **Research Director** (~2.5%) and **Manager** (~5%) show strong retention, likely reflecting seniority and equity benefits. Entry-level technical and sales roles need structured onboarding and career ladders.

---

## 7. Attrition by Income

### Business question: *Is compensation a differentiator between employees who stay and those who leave?*

Income is a direct lever HR and finance can adjust — worth examining before building predictive models.

In [ ]:
# Income bands for readable aggregation
income_bins = [0, 3000, 5000, 7500, 10000, 20000]
income_labels = ["< $3k", "$3k–5k", "$5k–7.5k", "$7.5k–10k", "$10k+"]
df["IncomeBand"] = pd.cut(
    df["MonthlyIncome"], bins=income_bins, labels=income_labels, right=False
)
income = attrition_summary(df, "IncomeBand")

fig = px.bar(
    income,
    x="IncomeBand",
    y="attrition_rate",
    text="attrition_rate",
    hover_data={"employees": True},
    labels={"attrition_rate": "Attrition Rate (%)", "IncomeBand": "Monthly Income Band"},
    title="Attrition Rate by Monthly Income Band",
    color="attrition_rate",
    color_continuous_scale=["#e76f51", "#2a9d8f"],
    range_color=[income["attrition_rate"].min(), income["attrition_rate"].max()],
)
fig.add_hline(y=overall_rate, line_dash="dash", line_color=OVERALL_RATE_COLOR)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(showlegend=False, coloraxis_showscale=False, yaxis_range=[0, 35])
fig.write_html(FIGURES_DIR / "eda_attrition_by_income.html")
fig.show()

income

**Interpretation:** Attrition is **inversely related to income**. Employees earning under **$3,000/month** leave at roughly double the company average; rates drop steadily as income rises. The highest-paid band ($10k+) shows the lowest turnover. Pay compression at the lower end is a plausible retention risk factor.

---

## 8. Attrition by Overtime

### Business question: *Does working overtime increase the likelihood of departure?*

Overtime is an operational policy HR can influence — unlike demographics, it is a direct management decision.

In [ ]:
ot_summary = attrition_summary(df, "OverTime")

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(ot_summary))
width = 0.35

stay_counts = [
    len(df[(df["OverTime"] == ot) & (df["Attrition_Yes"] == 0)])
    for ot in ot_summary["OverTime"]
]
leave_counts = [
    len(df[(df["OverTime"] == ot) & (df["Attrition_Yes"] == 1)])
    for ot in ot_summary["OverTime"]
]

ax.bar(x - width / 2, stay_counts, width, label="Stayed", color=COLOR_STAY)
ax.bar(x + width / 2, leave_counts, width, label="Left", color=COLOR_LEAVE)
ax.set_xticks(x)
ax.set_xticklabels(ot_summary["OverTime"])
ax.set_ylabel("Employee Count")
ax.set_xlabel("OverTime")
ax.set_title("Employee Count by Overtime and Attrition")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

for i, rate in enumerate(ot_summary["attrition_rate"]):
    ax.text(i, max(stay_counts[i], leave_counts[i]) + 25, f"{rate:.1f}% left", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_attrition_by_overtime.png", dpi=150, bbox_inches="tight")
plt.show()

ot_summary

**Interpretation:** Employees working **overtime leave at ~31%** — nearly **3× the rate** of those who do not (~10%). This is one of the strongest single-factor splits in the dataset. Reducing unsustainable overtime and monitoring burnout in overtime-heavy teams is a clear, actionable retention lever.

---

## 9. Attrition by Age

### Business question: *At what age are employees most likely to leave?*

Age bands help HR tailor programs — early-career development vs. mid-career compensation retention.

In [ ]:
age_bins = [18, 25, 30, 35, 40, 50, 61]
age_labels = ["18–24", "25–29", "30–34", "35–39", "40–49", "50–60"]
df["AgeBand"] = pd.cut(df["Age"], bins=age_bins, labels=age_labels, right=False)
age = attrition_summary(df, "AgeBand")

fig = px.bar(
    age,
    x="AgeBand",
    y="attrition_rate",
    text="attrition_rate",
    hover_data={"employees": True},
    labels={"attrition_rate": "Attrition Rate (%)", "AgeBand": "Age Band"},
    title="Attrition Rate by Age Band",
    color="attrition_rate",
    color_continuous_scale=["#2a9d8f", "#e76f51"],
    range_color=[age["attrition_rate"].min(), age["attrition_rate"].max()],
)
fig.add_hline(y=overall_rate, line_dash="dash", line_color=OVERALL_RATE_COLOR)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(showlegend=False, coloraxis_showscale=False, yaxis_range=[0, 28])
fig.write_html(FIGURES_DIR / "eda_attrition_by_age.html")
fig.show()

age

**Interpretation:** The **18–24** and **25–29** age bands show the highest attrition — consistent with early-career mobility. Rates decline through the 30s and 40s, then tick up slightly at **50–60** (possibly retirement-eligible transitions). Graduate retention programs and clear promotion timelines matter most for employees under 30.

---

## 10. Attrition by Job Satisfaction

### Business question: *Do dissatisfied employees leave at higher rates?*

Job satisfaction is a direct pulse on whether employees see a future at the company (scale 1 = lowest, 4 = highest).

In [ ]:
sat_labels = {1: "1 — Low", 2: "2", 3: "3", 4: "4 — High"}
sat = attrition_summary(df, "JobSatisfaction")
sat["label"] = sat["JobSatisfaction"].map(sat_labels)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    sat["label"],
    sat["attrition_rate"],
    color=COLOR_RATE,
    edgecolor="white",
)
ax.axhline(overall_rate, color=OVERALL_RATE_COLOR, linestyle="--", label=f"Avg ({overall_rate:.1f}%)")
ax.set_ylabel("Attrition Rate (%)")
ax.set_xlabel("Job Satisfaction Score")
ax.set_title("Attrition Rate by Job Satisfaction")
ax.set_ylim(0, 28)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

for bar, rate in zip(bars, sat["attrition_rate"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, f"{rate:.1f}%", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_attrition_by_satisfaction.png", dpi=150, bbox_inches="tight")
plt.show()

sat.drop(columns=["label"])

**Interpretation:** Employees with the **lowest job satisfaction (score 1)** leave at **~23%** — roughly double those with score 4 (~11%). The gradient is monotonic: each satisfaction step down corresponds to higher attrition. Regular satisfaction surveys and manager follow-ups on low scores have measurable ROI.

---

## 11. Attrition by Work-Life Balance

### Business question: *Does poor work-life balance drive employees away?*

Work-life balance (1 = worst, 4 = best) captures whether employees feel their personal time is respected.

In [ ]:
wlb_labels = {1: "1 — Poor", 2: "2", 3: "3", 4: "4 — Good"}
wlb = attrition_summary(df, "WorkLifeBalance")
wlb["label"] = wlb["WorkLifeBalance"].map(wlb_labels)

fig = px.bar(
    wlb,
    x="label",
    y="attrition_rate",
    text="attrition_rate",
    hover_data={"employees": True},
    labels={"attrition_rate": "Attrition Rate (%)", "label": "Work-Life Balance Score"},
    title="Attrition Rate by Work-Life Balance",
    color="attrition_rate",
    color_continuous_scale=["#2a9d8f", "#e76f51"],
    range_color=[wlb["attrition_rate"].min(), wlb["attrition_rate"].max()],
)
fig.add_hline(y=overall_rate, line_dash="dash", line_color=OVERALL_RATE_COLOR)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(showlegend=False, coloraxis_showscale=False, yaxis_range=[0, 38])
fig.write_html(FIGURES_DIR / "eda_attrition_by_worklifebalance.html")
fig.show()

wlb.drop(columns=["label"])

**Interpretation:** Employees rating work-life balance as **1 (Poor)** leave at **~31%** — the highest rate among satisfaction dimensions and nearly **2× the company average**. This aligns with the overtime finding: workload and boundary issues compound. Flexible scheduling and overtime caps directly address this signal.

---

## 12. Business Insights Summary

Key findings computed from this dataset. These inform feature selection, model evaluation priorities, and HR intervention design.

In [ ]:
# Compute headline metrics directly from the data
sales_rate = dept.loc[dept["Department"] == "Sales", "attrition_rate"].iloc[0]
rd_rate = dept.loc[dept["Department"] == "Research & Development", "attrition_rate"].iloc[0]
ot_yes = ot_summary.loc[ot_summary["OverTime"] == "Yes", "attrition_rate"].iloc[0]
ot_no = ot_summary.loc[ot_summary["OverTime"] == "No", "attrition_rate"].iloc[0]
role_top = role.sort_values("attrition_rate", ascending=False).iloc[0]
role_bot = role.sort_values("attrition_rate", ascending=True).iloc[0]
low_income_rate = income.loc[income["IncomeBand"] == "< $3k", "attrition_rate"].iloc[0]
wlb_poor = wlb.loc[wlb["WorkLifeBalance"] == 1, "attrition_rate"].iloc[0]
sat_low = sat.loc[sat["JobSatisfaction"] == 1, "attrition_rate"].iloc[0]
young_rate = age.loc[age["AgeBand"] == "18–24", "attrition_rate"].iloc[0]

insights = [
    f"Overall attrition is {overall_rate:.1f}% ({n_left} of {n_total} employees).",
    f"Sales department attrition ({sales_rate:.1f}%) exceeds R&D ({rd_rate:.1f}%) by {sales_rate - rd_rate:.1f} pp.",
    f"Overtime employees leave at {ot_yes:.1f}% vs {ot_no:.1f}% for non-overtime — a {ot_yes / ot_no:.1f}× difference.",
    f"Highest-risk role: {role_top['JobRole']} ({role_top['attrition_rate']:.1f}%). Lowest: {role_bot['JobRole']} ({role_bot['attrition_rate']:.1f}%).",
    f"Employees earning < $3k/month leave at {low_income_rate:.1f}% — well above the company average.",
    f"Poor work-life balance (score 1) attrition: {wlb_poor:.1f}%. Low job satisfaction (score 1): {sat_low:.1f}%.",
    f"Youngest band (18–24) attrition: {young_rate:.1f}%; median leaver age ({age_left:.0f}) is below median stayer age ({age_stay:.0f}).",
    "Top numeric correlates with attrition: tenure, job level, income, and stock options (negative); commute distance (positive).",
]

print("BUSINESS INSIGHTS")
print("=" * 60)
for i, line in enumerate(insights, 1):
    print(f"{i}. {line}")

pd.DataFrame({"insight": insights})

### Recommended HR actions (derived from EDA)

| Priority | Finding | Suggested action |
|----------|---------|------------------|
| High | Overtime ↔ 3× attrition | Cap overtime, monitor burnout in affected teams |
| High | Sales Representative ~40% attrition | Role-specific retention program, career pathing |
| High | Poor work-life balance → ~31% attrition | Flexible work policies, workload review |
| Medium | Low income → higher attrition | Compensation benchmarking for junior roles |
| Medium | Low job satisfaction → ~23% attrition | Manager training, stay interviews on low scores |
| Medium | Early-career age bands highest turnover | Mentorship, onboarding, promotion clarity |
| Low | Senior roles (Director, Manager) stable | Use as benchmark for retention best practices |

**Next step:** Feature engineering and model building — EDA signals suggest overtime, income, tenure, satisfaction scores, and role will be important predictors.